# Defensive Distillation

This code trains a teacher-student neural network model for knowledge distillation using the MNIST dataset (handwritten digit classification). The teacher model learns first, then its softened predictions (soft labels) are used to train a student model.
MNIST contains grayscale 28x28 images of digits (0-9)

This snippet sets up the foundation for implementing defensive distillation on the MNIST dataset by first handling data loading and preprocessing. The code imports TensorFlow/Keras and supporting libraries, then loads the MNIST handwritten digits dataset, which contains 60,000 training images and 10,000 test images. Each image is normalised to the range [0,1] by dividing by 255, which helps the neural network train more effectively. Since MNIST images are grayscale, the code adds a channel dimension (so each image becomes 28×28×1 rather than just 28×28), making them compatible with convolutional neural network layers. Finally, the labels are converted from integer digits (0–9) into one-hot encoded vectors using to_categorical, which allows the model to output probabilities for each of the 10 classes. This preprocessing step is essential groundwork before applying defensive distillation, where a model is trained at a higher "temperature" to smooth output probabilities and then distilled into a hardened student model that is more robust to adversarial perturbations.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import numpy as np

# Load and preprocess the MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = np.expand_dims(x_train, axis=-1)  # Add channel dimension
x_test = np.expand_dims(x_test, axis=-1)

y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

This part of the defensive distillation process defines and trains the teacher model. A simple convolutional neural network (CNN) is created using Keras’ Sequential API. The architecture includes two convolutional layers (32 and 64 filters respectively) each followed by max pooling for feature extraction and dimensionality reduction. After flattening, the network has a fully connected layer with 128 neurons and ReLU activation, and finally a 10-neuron softmax output layer for classifying digits 0–9. The model is compiled with the Adam optimiser and categorical cross-entropy loss, making it suitable for multi-class classification. Training is run for three epochs with a batch size of 128, using 10% of the training data for validation. This trained teacher model is the first step in defensive distillation: it will later be used to generate “soft” probability outputs (rather than just hard labels) that will guide the training of a more robust student model.

In [ ]:
# Define a simple CNN model
def create_model():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Train the teacher model
print("Training the teacher model...")
teacher_model = create_model()
teacher_model.fit(x_train, y_train, epochs=3, batch_size=128, validation_split=0.1)

This step produces the soft labels that lie at the heart of defensive distillation. Instead of using the teacher model’s standard hard predictions (one class with probability ≈1, others ≈0), the code introduces a temperature parameter to smooth the probability distribution. By dividing the logits by a higher temperature (here, 5.0) before applying the softmax, the predicted class probabilities become “softer” — meaning non-target classes retain non-zero values that reflect similarity between digits. For example, a “3” might be partly predicted as a “5” or “8,” rather than only as “3.” These softened labels carry richer information about class relationships, which will later be used to train the student model. This helps the student learn more general decision boundaries and reduces sensitivity to small adversarial perturbations, thereby improving robustness against adversarial attacks.

In [ ]:
# Generate soft labels using the teacher model
def generate_soft_labels(model, x_data, temperature=5.0):
    """
    Generates softened probability distributions by dividing logits by a temperature.
    :param model: Trained teacher model
    :param x_data: Input data
    :param temperature: Temperature for softening predictions
    :return: Soft labels (softened probabilities)
    """
    logits = model.predict(x_data)
    soft_labels = tf.nn.softmax(logits / temperature).numpy()
    return soft_labels

temperature = 5.0
soft_labels = generate_soft_labels(teacher_model, x_train, temperature)

This final stage of the defensive distillation pipeline trains and evaluates the student model. A new CNN with the same architecture as the teacher is created, but instead of being trained directly on the original hard labels, it is trained on the soft labels produced by the teacher at high temperature. This allows the student to learn smoother decision boundaries and capture the teacher’s knowledge about class similarities. After training for three epochs, both the teacher and student models are evaluated on the clean test set. Comparing their accuracies shows whether the distilled student retained predictive performance while gaining robustness. In practice, the student model may achieve similar accuracy to the teacher but be less vulnerable to adversarial examples, which is the primary goal of defensive distillation.

In [ ]:
# Train the student model using the soft labels
print("Training the student model...")
student_model = create_model()
student_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)
student_model.fit(x_train, soft_labels, epochs=3, batch_size=128, validation_split=0.1)

# Step 6: Evaluate both models on the clean test data
print("Evaluating the teacher model...")
teacher_loss, teacher_accuracy = teacher_model.evaluate(x_test, y_test, verbose=0)

print("Evaluating the student model...")
student_loss, student_accuracy = student_model.evaluate(x_test, y_test, verbose=0)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training the teacher model...


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 59s 135ms/step - accuracy: 0.8436 - loss: 0.5109 - val_accuracy: 0.9768 - val_loss: 0.0731
Epoch 2/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 67s 100ms/step - accuracy: 0.9787 - loss: 0.0661 - val_accuracy: 0.9890 - val_loss: 0.0424
Epoch 3/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 42s 100ms/step - accuracy: 0.9876 - loss: 0.0431 - val_accuracy: 0.9893 - val_loss: 0.0397
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step
Training the student model...
Epoch 1/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 45s 102ms/step - accuracy: 0.8480 - loss: 2.3014 - val_accuracy: 0.9878 - val_loss: 2.3007
Epoch 2/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 41s 96ms/step - accuracy: 0.9883 - loss: 2.3007 - val_accuracy: 0.9918 - val_loss: 2.3007
Epoch 3/3
422/422 ━━━━━━━━━━━━━━━━━━━━ 44s 104ms/step - accuracy: 0.9914 - loss: 2.3007 - val_accuracy: 0.9925 - val_loss: 2.3007
Evaluating the teacher model...
Evaluating the student model...


We print the accuracy of the teacher and student.

In [ ]:
print(f"Teacher model accuracy on clean test data: {teacher_accuracy * 100:.2f}%")
print(f"Student model accuracy on clean test data: {student_accuracy * 100:.2f}%")


Teacher model accuracy on clean test data: 98.63%
Student model accuracy on clean test data: 98.83%
